In [ ]:
!uv pip install -q unsloth datasets vllm

In [ ]:
from unsloth import FastLanguageModel
import torch


base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B-Base-bnb-4bit",
    max_seq_length = 1024,
    load_in_4bit = True,
    device_map = "auto",
)


==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-0.6b-base-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
print(f"Model embedding vocab size: {base_model.get_input_embeddings().weight.shape[0]}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

Model embedding vocab size: 151936
Tokenizer vocab size: 151670


In [ ]:
print(f" Difference between tokenizer vocab and model vocab: {base_model.get_input_embeddings().weight.shape[0] - len(tokenizer)}")
print(f" Length of tokenizer special tokens: {len(tokenizer.all_special_tokens)}")

 Difference between tokenizer vocab and model vocab: 266
 Length of tokenizer special tokens: 15


In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.all_special_tokens

['<|endoftext|>',
 '<|im_start|>',
 '<|im_end|>',
 '<|object_ref_start|>',
 '<|object_ref_end|>',
 '<|box_start|>',
 '<|box_end|>',
 '<|quad_start|>',
 '<|quad_end|>',
 '<|vision_start|>',
 '<|vision_end|>',
 '<|vision_pad|>',
 '<|image_pad|>',
 '<|video_pad|>']

In [ ]:
# 2. Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    base_model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["lm_head", "embed_tokens"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)


Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [ ]:
model.print_trainable_parameters()

trainable params: 321,257,472 || all params: 1,072,889,856 || trainable%: 29.9432


## Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("meta-math/GSM8K_zh")

Generating train split:   0%|          | 0/8792 [00:00<?, ? examples/s]

In [ ]:
def transform(x):
    start = "<|im_start|>"
    end = "<|im_end|>"
    think_start = '<think>'
    think_end = '</think>'

    question = x["question"]
    reasoning = x["answer"].split("####")[0]
    answer_only = x["answer_only"]

    prompt = f"{start}user\n{question}\n{end}\n{start}assistant\n{think_start}\n"

    response = (
        f"{reasoning}\n"
        f"{think_end}\n"
        f"Final answer: {answer_only}\n"
        f"{end}"
    )

    prompt_tokens = tokenizer(prompt, add_special_tokens=False)
    response_tokens = tokenizer(response, add_special_tokens=False)

    input_ids = prompt_tokens["input_ids"] + response_tokens["input_ids"]
    labels = ([-100] * len(prompt_tokens["input_ids"])) + response_tokens["input_ids"]

    input_ids = input_ids
    labels = labels
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
    }

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'answer_only', 'split', 'question_zh', 'answer_zh'],
        num_rows: 8792
    })
})

In [ ]:
for data in dataset["train"].select(range(5)):
    print("-"*50)
    print(f"Question: {data["question"]}")
    print(f"Reason: {data["answer"]}")
    print(f"Answer: {data["answer_only"]}")

--------------------------------------------------
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Reason: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72
Answer: 72
--------------------------------------------------
Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Reason: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10
Answer: 10
--------------------------------------------------
Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Reas

In [ ]:
transformed_dataset = dataset.map(transform)

Map:   0%|          | 0/8792 [00:00<?, ? examples/s]

In [ ]:
dataset["train"][0].keys()

dict_keys(['question', 'answer', 'answer_only', 'split', 'question_zh', 'answer_zh'])

In [ ]:
sample_data = transformed_dataset["train"].select(range(2)).select_columns(["input_ids","labels"])
for data in sample_data:
  print(f"Input_ids: {len(data['input_ids'])}")
  print(f"Labels: {len(data['labels'])}")

Input_ids: 114
Labels: 114
Input_ids: 111
Labels: 111


In [ ]:
from transformers import DataCollatorForSeq2Seq, AutoTokenizer

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
)

sample_data = collator(sample_data)

for k,v in sample_data.items():
  print(k,v.shape)

input_ids torch.Size([2, 114])
attention_mask torch.Size([2, 114])
labels torch.Size([2, 114])


### Split data

---



In [ ]:
from torch.utils.data import DataLoader
split_data = transformed_dataset["train"].train_test_split(test_size=0.2)
test_exact = split_data["test"].select(range(200))
split_torch_data= split_data.with_format("torch",columns=["input_ids","attention_mask","labels"])
split_data.set_format("torch", columns=["input_ids","attention_mask","labels"])
train_loader = DataLoader(split_torch_data["train"], batch_size=8, shuffle=True, collate_fn=collator)
val_loader = DataLoader(split_torch_data["test"], batch_size=8, shuffle=False, collate_fn=collator)

In [ ]:
def calculate_accuracy(y_preds, y_labels):
    preds_shifted = y_preds[:, :-1]
    labels_shifted = y_labels[:, 1:]


    mask = labels_shifted != -100


    valid_preds = preds_shifted[mask]
    valid_labels = labels_shifted[mask]


    if len(valid_labels) == 0:
        return 0.0
    return (valid_preds == valid_labels).sum().item() / len(valid_labels)

In [ ]:
import torch
import re
from tqdm.auto import tqdm

def evaluate_exact_match(model, tokenizer, validation_subset, device="cuda"):
    model.eval()
    correct_count = 0
    total_count = len(validation_subset)


    start = "<|im_start|>"
    end = "<|im_end|>"
    think_start = '<think>'
    think_end = '</think>'


    for item in tqdm(validation_subset, desc="Evaluating EM"):
        question = item["question"]
        true_answer = str(item["answer_only"]).strip()

        prompt = f"{start}user\n{question}\n{end}\n{start}assistant\n{think_start}\n"
        user_prompt = tokenizer(prompt, add_special_tokens=False, return_tensors="pt")

        inputs = {
            "input_ids": user_prompt["input_ids"].to(device),
            "attention_mask": user_prompt["attention_mask"].to(device)
        }


        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=0,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.convert_tokens_to_ids(end),
                do_sample=False
            )


        input_length = inputs["input_ids"].shape[1]
        generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=False)

        print("*" * 100)
        print(f"Generated text:\n{generated_text}")

        # 4. Extract the final answer
        if think_end in generated_text:
            post_think = generated_text.split(think_end)[-1]
            extracted_answer = post_think.split(end)[0].strip()
        else:
            extracted_answer = generated_text.split(end)[0].strip()


        numbers_in_output = re.findall(r'-?[\d,]+(?:\.\d+)?', extracted_answer)

        if numbers_in_output:
            predicted_number = numbers_in_output[-1].replace(',', '')
            clean_true_answer = true_answer.replace(',', '')
        else:
            predicted_number = extracted_answer
            clean_true_answer = true_answer

        print("-" * 50)
        print(f"Raw Extracted text: {extracted_answer}")
        print(f"Parsed Number: {predicted_number} | Actual answer: {true_answer}")

        if str(predicted_number) == clean_true_answer or true_answer in extracted_answer:
            correct_count += 1

    em_accuracy = correct_count / total_count if total_count > 0 else 0.0
    print(f"Exact Match (EM) Accuracy: {em_accuracy * 100:.2f}% ({correct_count}/{total_count})")

    return em_accuracy

In [ ]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

In [ ]:
from tqdm.auto import tqdm
from torch.cuda.amp import autocast

def train_loop(model, train_loader, optimizer, epoch, device, scheduler, val_loader=None, actual_match_data=None):
    accuracy = []
    losses = []
    val_accuracy = []
    val_losses = []
    actual_matich = []

    for e in range(epoch):
        train_loss = 0
        train_acc = 0
        model.train()
        for batch in tqdm(train_loader, desc=f"Epoch {e+1} [Train]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()


            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            y_preds = outputs.logits.argmax(dim=-1)
            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            train_loss += loss.item()
            train_acc += calculate_accuracy(y_preds, labels)

        losses.append(train_loss/len(train_loader))
        accuracy.append(train_acc/len(train_loader))
        print("-"*50)
        print(f"Epoch: {e+1}, Train Loss: {train_loss/len(train_loader):.4f}, Train Accuracy: {train_acc/len(train_loader):.4f}")

        if val_loader is not None:
            val_loss = 0
            val_acc = 0
            model.eval()
            for batch in tqdm(val_loader, desc=f"Epoch {e+1} [Val]"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                with torch.no_grad():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    loss = outputs.loss
                    y_preds = outputs.logits.argmax(dim=-1)
                    val_loss += loss.item()
                    val_acc += calculate_accuracy(y_preds, labels)
            val_losses.append(val_loss/len(val_loader))
            val_accuracy.append(val_acc/len(val_loader))
            print(f"Epoch: {e+1}, Val Loss: {val_loss/len(val_loader):.4f}, Val Accuracy: {val_acc/len(val_loader):.4f}")

        if actual_match_data is not None:
            actual_matich.append(evaluate_exact_match(model, tokenizer, actual_match_data.shuffle().select(range(2)), device))

    return losses, accuracy, val_losses, val_accuracy, actual_matich

## Pre trained model

In [ ]:
# evaluate_exact_match(base_model,tokenizer,split_data["test"].select(range(50)),model.device)

## Training

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCH = 1

optimizer = AdamW(model.parameters(), lr=1e-4)
num_training_steps = len(train_loader) * EPOCH

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_training_steps * 0.1,
    num_training_steps=num_training_steps,
)


In [ ]:
model.to("cuda")

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): ModulesToSaveWrapper(
          (original_module): Embedding(151936, 1024)
          (modules_to_save): ModuleDict(
            (default): Embedding(151936, 1024)
          )
        )
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
        

In [ ]:
train_loss, train_accuracy, val_loss, val_accuracy, exact_match = train_loop(
    model,
    train_loader,
    optimizer,
    EPOCH,
    model.device,
    scheduler,
    val_loader,
    test_exact
)

Epoch 1 [Train]:   0%|          | 0/880 [00:00<?, ?it/s]

Unsloth: Will smartly offload gradients to save VRAM!
--------------------------------------------------
Epoch: 1, Train Loss: 0.4588, Train Accuracy: 0.8747


Epoch 1 [Val]:   0%|          | 0/220 [00:00<?, ?it/s]

Epoch: 1, Val Loss: 0.4164, Val Accuracy: 0.8829


Evaluating EM:   0%|          | 0/2 [00:00<?, ?it/s]

****************************************************************************************************
Generated text:
The second dose has a wait time of 20/2=<<20/2=10>>10 minutes
So the total wait time is 20+10=<<20+10=30>>30 minutes

</think>
Final answer: 30
<|im_end|>
--------------------------------------------------
Raw Extracted text: Final answer: 30
Parsed Number: 30 | Actual answer: 30
****************************************************************************************************
Generated text:
There are 12 floors / 2 = <<12/2=6>>6 floors that are not full.
There are 12 floors - 6 floors = <<12-6=6>>6 floors that are at half-capacity.
There are 6 floors * 10 apartments = <<6*10=60>>60 apartments in the building.
There are 60 apartments * 4 people = <<60*4=240>>240 people in the building.

</think>
Final answer: 240
<|im_end|>
--------------------------------------------------
Raw Extracted text: Final answer: 240
Parsed Number: 240 | Actual answer: 360
Exact Match (EM) A

In [ ]:
split_data.reset_format()
evaluate_exact_match(model,tokenizer,split_data["test"].shuffle().select(range(50)),model.device)

Evaluating EM:   0%|          | 0/50 [00:00<?, ?it/s]

****************************************************************************************************
Generated text:
First find the total number of guppies the betta fish eat per day: 5 fish * 7 guppies/fish = <<5*7=35>>35 guppies
Then add the number of guppies the eel eats per day to find the total number of guppies per day: 35 guppies + 20 guppies = <<35+20=55>>55 guppies

</think>
Final answer: 55
<|im_end|>
--------------------------------------------------
Raw Extracted text: Final answer: 55
Parsed Number: 55 | Actual answer: 55
****************************************************************************************************
Generated text:
On Monday Carla counts the tiles 38 times, on Tuesday she counts the tiles twice in a row, so she counts them 38*2 = <<38*2=76>>76 times.
On Tuesday Carla counts the books 75 times, on Tuesday she counts the books 3 times in a row, so she counts them 75*3 = <<75*3=225>>225 times.
In total Carla has counted 76+225 = <<76+225=291>>291 times o

0.5

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub("Blankyy/reasoning-math-model")
tokenizer.push_to_hub("Blankyy/reasoning-math-model")

Saved model to https://huggingface.co/Blankyy/reasoning-math-model
